In [ ]:
import sys

if "google.colab" in sys.modules:
    import os

    repo = "/content/lmbike_workshop_gimbal"
    sim = "/content/lmbike_workshop_gimbal/bike_mujoco_sim"
    if not os.path.isdir(repo):
        !git clone https://github.com/innowing-robotics-interns/lmbike_workshop_gimbal.git /content/lmbike_workshop_gimbal
    %cd /content/lmbike_workshop_gimbal/bike_mujoco_sim
    if sim not in sys.path:
        sys.path.insert(0, sim)

    # Colab's Jupyter must stay the copy it shipped with. requirements.txt
    # does not list Jupyter. The constraint pins whatever Jupyter packages
    # are already installed so pip cannot replace them.
    import importlib.metadata as metadata
    protected = {
        "notebook", "jupyter", "jupyterlab", "jupyter-client", "jupyter-core",
        "jupyter-server", "ipykernel", "ipython", "nbformat", "nbclient",
        "nbconvert", "nbclassic",
    }
    pinned = []
    for dist in metadata.distributions():
        name = dist.metadata["Name"]
        if name and name.lower().replace("_", "-") in protected:
            pinned.append(f"{name}=={dist.version}")
    with open("/tmp/colab-jupyter-constraints.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(pinned) + "\n")
    !pip install -q --upgrade-strategy only-if-needed -r requirements.txt -c /tmp/colab-jupyter-constraints.txt


# Lesson 4. Driving the bicycle while it balances

You now have a balancing loop with gains that keep the bicycle upright while it rides. In this lesson you become the driver. You will tell the bicycle to speed up, slow down and turn, while the balancing loop keeps working in the background to stop it from falling. You will also play the part of the outside world, and shove the bicycle sideways to see whether the loop can catch it.

Both of these test the same idea. The balancing loop's whole job is to keep the lean near 0°. Anything that pushes the lean away from 0° is a **disturbance**, and this lesson tries two very different kinds:

- **A sideways push** is a sudden disturbance from outside, like a gust of wind or someone bumping into the bicycle. Nobody asked for it; it just happens.
- **A turn** is a disturbance that you ask for yourself. Turning needs steering, and a turning bicycle naturally leans into the turn, the way a cyclist leans into a corner. So every turn request disturbs the very thing the balancing loop is trying to hold steady.

Can the loop keep the bicycle up through both? That is what you will test.

You can drive with the arrow keys, with an on-screen stick, or with a gamepad if you have one.

## Learning objectives

By the end of this lesson you will be able to:

1. Describe what pushing the stick up, down, left and right asks the bicycle to do.
2. Read a lean graph of a sideways push, and describe how the balancing loop catches it.
3. Explain why a turn request is also a disturbance for a balancing loop whose job is to keep the lean near 0°.
4. Compare the lean graphs of a ride with the stick centered and a ride with the stick held to one side.

## Glossary

| Term | Meaning |
| --- | --- |
| Teleoperation | Driving a machine from a distance with a controller, like a remote-control car or a drone. In this lesson you drive, while the balancing loop keeps running on its own. |
| Disturbance | Anything that pushes a controlled system away from its target. A gust of wind or a bump in the road would be disturbances. In this lesson, a sideways push and your own turn requests are both disturbances, because they make the bicycle lean away from upright. |
| Disturbance rejection | How well a controller holds its target despite disturbances. For us, it means how small the lean stays, and how quickly it returns to near 0°, after a push or during a turn. |
| Sideways push | A short, sharp shove on the side of the frame. The Strength box sets how hard, in m/s. It starts at 0.8. The program adds that sideways speed once and then lets go, like a hand slapping the side of the bicycle. |
| Deadzone | A small zone around the center of the stick where movement is ignored. Without it, the tiny shake of your hand on a resting stick would keep nudging the speed. |
| Cruise speed | The normal forward speed, about 2.2 m/s. It is the same `target_speed` as in Lesson 1, and the bicycle uses it whenever the stick is centered. |
| Heading | The direction the bicycle is pointing, like a compass direction. |

## Step 1. Start the lesson

As in the earlier lessons, the first code cell prepares the notebook. This time it also loads the code that reads the stick. Run it now, and continue when you see the line that begins with `project_root:`.

In [1]:
import os
import sys

def choose_mujoco_gl(platform, display, existing):
    if existing:
        return existing
    if str(platform).startswith("linux") and not display:
        return "egl"
    return None

chosen = choose_mujoco_gl(sys.platform, os.environ.get("DISPLAY"), os.environ.get("MUJOCO_GL"))
if chosen and "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = chosen

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace

from sim.config import load_config
from sim.loop import make_sim, run_headless, summarize
from sim.pid_controller import RollSteerPID, apply_speed_refs

np.set_printoptions(precision=4, suppress=True)
print("project_root:", project_root)
print("platform:", sys.platform, "MUJOCO_GL:", os.environ.get("MUJOCO_GL", "(unset)"))


from sim.joystick import speed_goal_from_stick


project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
platform: linux MUJOCO_GL: (unset)


## Step 2. Learn the controls, then check the stick's speeds

The stick works like the one on a game controller. It springs back to the center when you let go, and the center means "no extra request".

- **Stick up or down (or the Up and Down arrow keys) sets the speed.** With the stick centered, the bicycle rides at the cruise speed of about 2.2 m/s. Pushing the stick up asks for a higher speed, up to the maximum in the settings. Pulling it down asks for a lower speed, and pulled all the way down it asks for a slow backwards speed. Small movements inside the deadzone, near the center, are ignored.
- **Stick left or right (or the Left and Right arrow keys) asks the bicycle to turn.** While you hold it to one side, a turning request is added on top of the balancing loop's own steering. When you let go, the bicycle notes the direction it is now pointing, its heading, and a small extra loop steers gently to keep it going that way.
- **Key 0 asks for a speed of 0.** Remember from Lesson 1 that steering can only balance a bicycle that is moving, so a bicycle told to stop will fall over. Move the stick away from the center to get going again.
- **Space, or the Reset button in the stick window, stands the bicycle back up** after a fall.

Throughout all of this, the balancing loop stays switched on. Your requests are added on top of it. They never replace it.

The next cell loads gains that can keep the bicycle up. If you saved your tuned gains (or the reference set) into `configs/tutorial.yaml` at the end of Lesson 3, it uses that file. If `tutorial.yaml` still holds the weak starting gains, it loads `configs/new_bike.yaml` instead, so this lesson works either way. The first printed line tells you which file was used.

The cell then asks the program what speed three stick positions would request. Run the cell and check that you see:

- `None` for the centered stick. This means "no speed request from the stick", so the bicycle follows the normal speed schedule up to the cruise speed.
- A positive number, larger than the cruise speed, for full forward.
- A negative number, the slow backwards speed, for full back.

In [2]:
# Prefer a tuned panel. Fall back to new_bike.yaml if tutorial.yaml is still weak.
tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
    print("loaded tuned gains from", os.path.relpath(new_bike_yaml, project_root))
else:
    print("loaded", os.path.relpath(tutorial_yaml, project_root))
print(
    "outer gains kp={:.4f} ki={:.4f} kd={:.4f} kp_steer={:.4f}".format(
        cfg.pid_kp_roll, cfg.pid_ki_roll, cfg.pid_kd_roll, cfg.pid_kp_steer
    )
)
print("center stick speed goal", speed_goal_from_stick(0.0, cfg))
print("full forward", speed_goal_from_stick(1.0, cfg))
print("full back", speed_goal_from_stick(-1.0, cfg))


loaded tuned gains from configs/new_bike.yaml
outer gains kp=7.5356 ki=1.2615 kd=11.2605 kp_steer=1.1164
center stick speed goal None
full forward 3.5
full back -1.0


## Step 3. Drive the bicycle yourself (optional)

This step is for fun, and for getting a feel for the controls. Step 5 records the important comparison as graphs, so if you are short of time, go straight to Step 5.

The next cell opens two windows: the 3D view of the bicycle, and a small window with an on-screen stick. To use the stick, drag the orange knob with the mouse. It springs back to the center when you let go. The stick window also shows the bicycle's current lean in degrees. You can use the arrow keys while the 3D window or the stick window is selected. Hold a key and the request grows. Let go and it returns to center quickly.

1. Run the cell, and let the bicycle ride for a few seconds with the stick centered.
2. Hold the stick to the left for a moment, then let go. Watch how far the bicycle leans as it turns, and what it does after you let go.
3. Try a few other moves: speed up, slow down, and turn the other way.
4. Click the **Push sideways** button in the stick window while the bicycle is riding. Step 4 looks at this push closely.
5. When you have finished, close the MuJoCo window. The cell finishes once that window is closed.

If the cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run this cell again, and run the Step 1 and Step 2 cells once more before you continue to Step 5.

In [3]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
sim = make_sim(model_spec="new_bike", mode="remote", cfg=cfg)
print("Remote stick: center = cruise + hold heading. Close MuJoCo when done.")
run_viewer(sim)
print("Window closed.")


project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
MUJOCO_GL: glfw
Remote stick: center = cruise + hold heading. Close MuJoCo when done.
Remote stick (floating window, or hold arrows — they grow, then spring back):
  Up/Down     speed around cruise
  Left/Right  turn rate
  0           stop
  Space / Reset button  reset upright
  Push sideways button  a shove from the side; Strength sets how hard
Stick centered: cruise at 2.20 m/s and hold heading.
Window closed.


## Step 4. Push the bicycle sideways

So far, every disturbance in this course came from the bicycle itself. Now you will add one from the outside: a short, sharp shove on the side of the frame, like someone slapping the bicycle once as it rides past. Nobody asked the bicycle to do anything. It is simply knocked off balance, and the balancing loop has to notice and catch it on its own.

When you click **Push sideways**, the program gives the bicycle's frame a sudden sideways speed. How hard is the number in the Strength box next to the button. It starts at 0.8 m/s. After that instant, the balancing loop is in charge again, exactly as before.

There are two cells:

1. The **window cell** opens the 3D view and the stick window. Leave the stick centered, wait until the bicycle is cruising, then click **Push sideways**. Watch the lean number in the stick window jump, and then watch what happens to it over the next second. Wait for the bicycle to settle, then push again a few times. When you have finished, close the MuJoCo window.
2. The **graph cell** repeats one push without a window, so that you can see it as a line. It rides for 4 seconds with the normal speed schedule, gives exactly the same push at the 2-second mark, and draws the lean with a dotted vertical line at the moment of the push. The ride uses the plain speed schedule, so the line starts from almost exactly upright and the push is easy to spot. It also prints the lean just before the push, the largest lean after it, and either the lean at the end of the ride or the time of a fall.

Read the graph in three parts: before the dotted line, the moment just after it, and the rest of the ride. How far did the push knock the bicycle over? How long did the balancing loop take to bring it back near 0°? Did it overshoot to the other side on the way back?

The graph cell has a `push_speed` setting at the top. Leave it at `0.8` for now, the same number as the Strength box. You will change it in the Questions.

If the window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [4]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
sim = make_sim(model_spec="new_bike", mode="remote", cfg=cfg)
print("Leave the stick centered and click Push sideways a few times.")
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")

project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
MUJOCO_GL: glfw
Leave the stick centered and click Push sideways a few times.
Close the MuJoCo window to finish this cell.
Remote stick (floating window, or hold arrows — they grow, then spring back):
  Up/Down     speed around cruise
  Left/Right  turn rate
  0           stop
  Space / Reset button  reset upright
  Push sideways button  a shove from the side; Strength sets how hard
Stick centered: cruise at 2.20 m/s and hold heading.
Window closed.


In [ ]:
from sim.loop import shove_bicycle_sideways, step_physics

# How hard the push is: the sideways speed it gives the frame, in m/s.
# The Push sideways button uses 0.8.
push_speed = 0.8
push_time = 2.0
ride_seconds = 4.0

tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
cfg = replace(cfg, duration=ride_seconds)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)

pushed = False
while float(sim.data.time) < ride_seconds:
    if not pushed and float(sim.data.time) >= push_time:
        shove_bicycle_sideways(sim, push_speed)
        pushed = True
    step_physics(sim)

time_s = np.asarray(sim.history["time"], dtype=float)
roll_deg = np.degrees(np.asarray(sim.history["roll"], dtype=float))
after = time_s >= push_time
tip = np.where(np.abs(roll_deg) > 45.0)[0]
print(f"push speed = {push_speed:.2f} m/s at t = {push_time:.1f} s")
print(f"lean just before the push = {roll_deg[~after][-1]:+.2f} deg")
print(f"largest lean after the push = {float(np.max(np.abs(roll_deg[after]))):.1f} deg")
if len(tip):
    print(f"Result: fell. The lean passed 45 deg at {time_s[tip[0]]:.2f} s.")
else:
    print(f"Result: held. Lean at the end of the ride = {roll_deg[-1]:+.2f} deg")

fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
ax.plot(time_s, roll_deg)
ax.axvline(push_time, color="C3", linestyle=":", label="sideways push")
ax.axhline(0.0, color="0.7", linewidth=0.8)
ax.axhline(45.0, color="0.5", linestyle="--", linewidth=0.8)
ax.axhline(-45.0, color="0.5", linestyle="--", linewidth=0.8)
ax.legend()
ax.set_xlabel("time (s)")
ax.set_ylabel("bicycle lean (deg)")
ax.set_title(f"push speed = {push_speed} m/s")
plt.show()

## Step 5. Compare a centered stick with a held turn

Now for the main experiment. You will compare two rides that are identical except for the stick. In one ride the stick stays centered the whole time. In the other it is held to the left the whole time. Both rides use the gains that can keep the bicycle up, and both use the driving mode, which the program calls `remote` mode.

There are three cells:

1. **First window cell**: leave the on-screen stick centered for the whole ride. Watch, then close the window.
2. **Second window cell**: hold the stick to the left for the whole ride. Watch, then close the window.
3. **Graph cell**: repeats both rides without a window, holding the stick perfectly still for you. In the code, the stick's left-right position is called `stick_x`: `stick_x = 0` is centered, and `stick_x = 1` is held fully left. The cell prints a short report for each ride, then draws the two lean graphs side by side.

Each report lists a few numbers. The two to look at are `max |roll|`, the largest lean during the ride, and `mean |roll|`, the average size of the lean over the whole ride. `final speed` is how fast the bicycle was going at the end.

When you read the two graphs, look at more than the highest point. Does each line settle down after the start? Where does it settle: right on 0°, or at a steady few degrees to one side? And how far is each line from the 45° fall mark?

If a window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cells again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [6]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
cfg = replace(cfg, duration=4.0)
sim = make_sim(model_spec="new_bike", mode="remote", cfg=cfg)
print("Leave the stick centered. Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")


project_root: /home/lmbike2/bike-workshop/bike_mujoco_sim
MUJOCO_GL: glfw
Leave the stick centered. Close the MuJoCo window to finish this cell.
Remote stick (floating window, or hold arrows — they grow, then spring back):
  Up/Down     speed around cruise
  Left/Right  turn rate
  0           stop
  Space / Reset button  reset upright
  Push sideways button  a shove from the side; Strength sets how hard
Stick centered: cruise at 2.20 m/s and hold heading.
Window closed.


In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
cfg = replace(cfg, duration=4.0)
sim = make_sim(model_spec="new_bike", mode="remote", cfg=cfg)
print("Hold the stick to the left. Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")


In [ ]:
def run_remote(stick_x, seconds=4.0):
    sim = make_sim(model_spec="new_bike", mode="remote", cfg=replace(cfg, duration=seconds))
    sim.goals.stick_x = float(stick_x)
    history = run_headless(sim, duration=seconds)
    return sim, history

straight_sim, straight = run_remote(0.0)
turn_sim, turning = run_remote(1.0)
summarize(straight, model_path=straight_sim.model_path, mode="remote center")
summarize(turning, model_path=turn_sim.model_path, mode="remote stick_x=1")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True, constrained_layout=True)
for ax, hist, title in (
    (axes[0], straight, "stick_x = 0"),
    (axes[1], turning, "stick_x = 1"),
):
    ax.plot(hist["time"], np.degrees(hist["roll"]))
    ax.axhline(0.0, color="0.7", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("roll (deg)")
plt.show()

turn_roll = float(np.max(np.abs(np.degrees(turning["roll"]))))
print(f"max |roll| while turning = {turn_roll:.1f} deg")
if turn_roll > 30.0:
    raise RuntimeError("turn leaned too far; check that tuned gains are loaded")


## Step 6. Ride the speed schedule for comparison

Finally, ride the bicycle the way you did in Lessons 1 to 3: no stick at all, just the speed schedule, speeding up and then cruising. This gives you a baseline to compare the driving rides against.

Run the window cell and close the window when the ride ends. Then run the graph cell. This time the graph has two lines sharing one time axis:

- The **blue line** is the lean. Read it with the scale on the left.
- The **orange line** is the bicycle's forward speed. Read it with the scale on the right.

Put this lean line side by side with the two lean lines from Step 5. How does riding with no stick compare with driving?

If the window cell stops with a message saying MuJoCo was already imported, use **Kernel → Restart Kernel**. Run the window cell again, then run the Step 1 and Step 2 cells once more before you run the graph cell.

In [ ]:
import os
import sys

def find_project_root(start=None):
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(current, "run_sim.py")) and os.path.isfile(
            os.path.join(current, "sim", "__init__.py")
        ):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError(
        "Could not find bike_mujoco_sim. Start Jupyter from the folder that "
        "contains run_sim.py, or from its tutorials folder."
    )

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Colab has no desktop. Keep the egl backend from the first cell and let
# run_viewer draw the ride in the page. A local window still needs glfw.
if "google.colab" in sys.modules:
    os.environ.setdefault("MUJOCO_GL", "egl")
else:
    if "mujoco" in sys.modules and os.environ.get("MUJOCO_GL") not in (None, "", "glfw"):
        raise RuntimeError(
            "MuJoCo was already imported with MUJOCO_GL=%r. Restart the kernel, "
            "then run this cell again." % os.environ.get("MUJOCO_GL")
        )
    os.environ["MUJOCO_GL"] = "glfw"

from dataclasses import replace
from sim.config import load_config
from sim.loop import make_sim
from sim.viewer import run_viewer

print("project_root:", project_root)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
cfg = replace(cfg, duration=6.0)
sim = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=cfg)
print("speed schedule, duration", cfg.duration)
print("Close the MuJoCo window to finish this cell.")
run_viewer(sim)
print("Window closed.")


In [ ]:
cruise = make_sim(model_spec="new_bike", mode="speed-schedule", cfg=replace(cfg, duration=6.0))
cruise_hist = run_headless(cruise, duration=6.0)
summarize(cruise_hist, model_path=cruise.model_path, mode="speed-schedule")
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ax.plot(cruise_hist["time"], np.degrees(cruise_hist["roll"]), label="roll")
ax.set_ylabel("roll (deg)")
ax.set_xlabel("time (s)")
ax2 = ax.twinx()
ax2.plot(cruise_hist["time"], cruise_hist["speed"], color="C1", label="speed")
ax2.set_ylabel("speed (m/s)")
ax.set_title("speed schedule")
plt.show()


## Questions

These questions check whether you can predict how the bicycle reacts to a disturbance, first a push and then a turn. Write your answers down before you run each cell.

### Pushes

1. In Step 4, the 0.8 m/s push was caught. Predict a push half as strong, `push_speed = 0.4`. Will the largest lean after the push be about half as big, about the same, or bigger?
2. Now predict a push only a little stronger, `push_speed = 1.0`. Will the balancing loop still catch it?

To check, go back to the Step 4 graph cell, change `push_speed` to `0.4` and run it, then change it to `1.0` and run it again. When you are done, set it back to `0.8`.

### Turns

3. In Step 5, compare the two rides once they had settled after the start. Which one leaned more: the centered stick, or the stick held left? Was the difference large or small compared with the 45° fall mark?
4. Predict what happens with the stick held fully to the **right** (`stick_x = -1`). Will the largest lean stay under 30°? Will the bicycle settle leaning to the same side as in the left turn, or to the other side?

The next cell runs a 4-second ride with the stick held still at the position you put in `stick_try`. It prints the largest lean, the average lean after the first 1.5 seconds (the sign tells you which side the bicycle settled on), and a result line. Remember that `+1` is fully left, `-1` is fully right, `0` is centered, and numbers in between are partway.

1. Run it with `stick_try = -1.0` (fully right) and compare the result with your prediction.
2. Run it with `0.5` (halfway left).
3. Run it with `0` (centered), and compare the result with the Step 5 rides.

In [ ]:
# Edit stick_try and run again. Trials: -1, then 0.5, then 0
stick_try = -1.0

tutorial_yaml = os.path.join(project_root, "configs", "tutorial.yaml")
new_bike_yaml = os.path.join(project_root, "configs", "new_bike.yaml")
cfg = load_config(tutorial_yaml)
if cfg.pid_kd_roll < 8.0 or cfg.pid_kp_roll < 6.0:
    cfg = load_config(new_bike_yaml)
sim = make_sim(model_spec="new_bike", mode="remote", cfg=replace(cfg, duration=4.0))
sim.goals.stick_x = float(stick_try)
history = run_headless(sim, duration=4.0)
max_roll = float(np.max(np.abs(np.degrees(history["roll"]))))
print(f"stick_x = {float(stick_try):+.1f}")
print(f"max |roll| = {max_roll:.1f} deg")
settled = np.degrees(np.asarray(history["roll"], dtype=float))[np.asarray(history["time"]) > 1.5]
print(f"average lean after the first 1.5 s = {float(np.mean(settled)):+.1f} deg")
if max_roll > 45.0:
    print("Result: fell.")
elif max_roll > 30.0:
    print("Result: stayed under 45 deg, and leaned past 30 deg.")
else:
    print("Result: held. Maximum |roll| stayed under 30 deg.")

## Discussion

Here are some questions students often ask after this lesson, with explanations.

### How did the balancing loop catch the push?

The push gave the frame a sudden sideways speed, so the bicycle started tipping fast. That fast tip is exactly what the D part reacts to, so the loop answered with a strong steering command almost at once, steering the wheels back under the falling bicycle. As the lean grew, P added its own push back toward upright. Within about a tenth of a second the lean stopped growing and turned around. On the way back it swung a few degrees past 0° to the other side, because the loop was still steering hard as the bicycle passed upright, and then it settled back toward 0° over the next second or two. You did nothing: the loop noticed the disturbance through its lean measurement and corrected it on its own.

### Why did a slightly stronger push knock the bicycle over, when a weaker one was barely a problem?

Every balancing loop has a limit. The steering command is capped at its maximum steer rate, and the handlebars can only move the wheels so far, so fast. A small push leaves plenty of room inside those limits, and the loop catches it easily. A bigger push needs more correction, and there comes a point where even the maximum steering cannot get the wheels back under the bicycle in time. Past that point the lean keeps growing and the bicycle falls. On this bicycle the limit sits somewhere between 0.8 m/s and 1.0 m/s, which is why the result changed so suddenly between those two pushes.

### Why could a turn knock the bicycle over?

To turn, the handlebars have to be steered away from straight, and a turning bicycle leans into the turn. Both of these work against what the balancing loop is trying to do, which is to keep the lean at 0° using the handlebars. So a turn request is a disturbance: left alone, it would push the bicycle farther from upright.

With well-tuned gains, the balancing loop absorbs that push and keeps the lean small. That is what disturbance rejection means. With weak gains, like the ones in Lesson 2, a turn could easily be the push that tips the bicycle over.

### Did holding the stick switch the balancing loop off?

No. The balancing loop runs for the whole ride, whatever you do with the stick. The stick only adds requests on top of it: a speed request from up and down, and a turning request from left and right.

This is why you can drive freely without balancing the bicycle yourself, just as a rider with good balance can think about where to go rather than about how to stay up.

### Why didn't the lean settle exactly at 0°, even with the stick centered?

You probably noticed that neither ride ended up sitting exactly on the 0° line. Each one settled at a steady lean of a few degrees and stayed there.

If you watch the window closely, you will see why: the bicycle is not riding in a perfectly straight line. Even with the stick centered, it curves as it rides, while the heading loop and the balancing loop share the steering. A curving bicycle has to lean into its curve to stay balanced, so a small steady lean is the result.

The important result is the size of the lean. Every turning ride stayed far below the 45° fall mark, which means the balancing loop rejected the disturbance, just as it rejected the 0.8 m/s push in Step 4.